# Create RGS Awards (Royal Geographical Society with IBG)

Creates Royal Geographical Society (with the Institute of British Geographers)
grants from the Society's own "Projects supported" page,
https://www.rgs.org/exploration/grants/projects-supported ("Grants awarded in
YYYY": one heading per scheme, one bullet per award with recipient(s),
institution and project title). The live page shows only the current round;
earlier rounds come from Internet Archive captures of the same page.
**152 awards: 2025 (73) and 2026 (79) rounds; 100% title and lead recipient,
91% institution, no amounts** (local run 2026-10-01).

Schemes kept (fieldwork and expedition research, per the batch brief): Small
Research Grants, Postgraduate Research Awards, Monica Cole, Henrietta Hutton,
Gilchrist Fieldwork, Geographical Fieldwork Grants, Ralph Brown Expedition,
Gino Watkins Fund, Walters Kundert, Thesiger-Oman, RGS Explore, Neville Shulman
Challenge, Ran and Ginny Fiennes, Ray Y Gildea Jr, Peter Smith, Jasmin Leila,
Hong Kong Research Grant, Geographical Club Award, Frederick Soddy Postgraduate,
Fieldwork Apprenticeships, and Journey in Audio (**flagged**: place-based audio
projects, kept under "when unsure, keep"). **Excluded** in the script: Frederick
Soddy Schools Award and Innovative Teaching Grant (22 grants to schoolteachers
for pupils' fieldwork: education, not research).

A project that won two schemes in one round (e.g. Geographical Fieldwork Grant +
Peter Smith Award) is two awards. Team awards: first named recipient is the lead,
the second the co-lead, all in `investigators`.

**Prerequisites:**
- Run `scripts/local/rgs_to_s3.py` first. It uploads
  `s3://openalex-ingest/awards/rgs/rgs_projects.parquet` (§1.4 shrink guard).

**`funder_award_id` (§2.1.1):** RGS does not publish its grant references.
Citing works quote them (e.g. "SRG 23.13", "PRA 13.24", "FSPA 05.21": scheme,
number, year), but the public lists carry no number, so nothing can be matched
without guessing. Synthetic key `RGS-{year}-{scheme-code}-{lead-slug}`, scheme
codes following RGS's own abbreviations (SRG, PRA, FSPA, RBEA, WKF ...).
Collapses onto 0 of 204 citation shells.

**Dates:** start_year = end_year = award round; start/end dates NULL.
**Amounts:** not published per award (NULL; currency NULL).

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320320195`
- display_name: Royal Geographical Society
- ror_id / doi: from `openalex.funders.funders` (ror 03k7szx23, doi 10.13039/501100000623)

**Priority:** `599` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.rgs_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/rgs/rgs_projects.parquet`;

In [ ]:
%sql
SELECT year, scheme, COUNT(*) as n FROM openalex.awards.rgs_raw GROUP BY 1, 2 ORDER BY 1, 3 DESC;

In [ ]:
%sql
DESCRIBE openalex.awards.rgs_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.rgs_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320320195 is a Crossref-registered funder, so it MUST resolve to exactly 1
row in `openalex.funders.funders`. If 0 rows, STOP (do not proceed) and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320320195;

## Step 2: Create RGS Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.rgs_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320320195  -- Royal Geographical Society
),

base AS (
    -- struct field order MUST match openalex_awards_raw: ...orcid, role_start, affiliation.
    SELECT
        g.*,
        named_struct(
            'given_name', NULLIF(TRIM(g.lead_given_name), ''),
            'family_name', NULLIF(TRIM(g.lead_family_name), ''),
            'orcid', CAST(NULL AS STRING),
            'role_start', CAST(NULL AS DATE),
            'affiliation', named_struct(
                'name', NULLIF(TRIM(g.institution), ''),
                'country', CAST(NULL AS STRING),
                'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
            )
        ) AS lead_struct,
        CASE WHEN NULLIF(TRIM(g.co_family_name), '') IS NOT NULL THEN named_struct(
            'given_name', NULLIF(TRIM(g.co_given_name), ''),
            'family_name', NULLIF(TRIM(g.co_family_name), ''),
            'orcid', CAST(NULL AS STRING),
            'role_start', CAST(NULL AS DATE),
            'affiliation', named_struct(
                'name', NULLIF(TRIM(g.co_institution), ''),
                'country', CAST(NULL AS STRING),
                'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
            )
        ) END AS co_struct,
        transform(
            from_json(g.investigators_json, 'array<struct<given:string,family:string,inst:string>>'),
            x -> named_struct(
                'given_name', NULLIF(TRIM(x.given), ''),
                'family_name', NULLIF(TRIM(x.family), ''),
                'orcid', CAST(NULL AS STRING),
                'role_start', CAST(NULL AS DATE),
                'affiliation', named_struct(
                    'name', NULLIF(TRIM(x.inst), ''),
                    'country', CAST(NULL AS STRING),
                    'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
                )
            )
        ) AS team_arr
    FROM openalex.awards.rgs_raw g
    WHERE g.funder_award_id IS NOT NULL AND TRIM(g.funder_award_id) != ''
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    NULLIF(TRIM(g.title), '') as display_name,
    CAST(NULL AS STRING) as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    CAST(NULL AS DOUBLE) as amount,
    CAST(NULL AS STRING) as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    CASE WHEN g.scheme RLIKE '(?i)fellowship' THEN 'fellowship' ELSE 'research' END as funding_type,
    CASE WHEN NULLIF(TRIM(g.sub_award), '') IS NOT NULL THEN CONCAT(g.scheme, ' (', TRIM(g.sub_award), ')')
         ELSE g.scheme END as funder_scheme,
    'rgs_projects_supported' as provenance,
    CAST(NULL AS DATE) as start_date,
    CAST(NULL AS DATE) as end_date,
    TRY_CAST(g.year AS INT) as start_year,
    TRY_CAST(g.year AS INT) as end_year,
    g.lead_struct as lead_investigator,
    g.co_struct as co_lead_investigator,
    CASE WHEN size(g.team_arr) > 0 THEN g.team_arr ELSE ARRAY(g.lead_struct) END as investigators,
    g.source_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM base g
CROSS JOIN src_funder f;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'rgs_projects_supported' AND priority = 599;

-- Insert into openalex_awards_raw with priority.
-- Priority 599: direct-from-funder ingest of the RGS-IBG grants programme lists.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    599 as priority
FROM openalex.awards.rgs_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_awards, COUNT(DISTINCT id) as distinct_ids,
       (SELECT COUNT(*) FROM openalex.awards.rgs_raw) as raw_rows
FROM openalex.awards.rgs_awards;

In [ ]:
%sql
-- §2.1.1 shape check: every key is the synthetic RGS-{year}-{scheme}-{slug} form.
SELECT
    SUM(CASE WHEN funder_award_id RLIKE '^RGS-20[0-9]{2}-[A-Z0-9-]+-[a-z0-9-]+$' THEN 1 ELSE 0 END) as good_shape,
    SUM(CASE WHEN funder_award_id NOT RLIKE '^RGS-20[0-9]{2}-[A-Z0-9-]+-[a-z0-9-]+$' THEN 1 ELSE 0 END) as bad_shape
FROM openalex.awards.rgs_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, funding_type, start_year,
       lead_investigator.given_name, lead_investigator.family_name,
       lead_investigator.affiliation.name, size(investigators) as n_inv
FROM openalex.awards.rgs_awards LIMIT 20;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt FROM openalex.awards.rgs_awards GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- Section 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.rgs_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.rgs_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.7 coverage: RGS publishes no per-award amounts (NULL by design).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(amount) as has_amount,
    COUNT(start_year) as has_start_year,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_institution,
    COUNT(co_lead_investigator) as has_co_lead
FROM openalex.awards.rgs_awards;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'rgs_projects_supported'
GROUP BY provenance, priority;